# Team Member Contribution Table

| Member      | Contribution |
| ----------- | ------------ |
| Petter (SM) | See classification response of alternative classifiers section  |
| Kim         | See ML SVM Classification  |
| Elliot      | Working on... |
| Agust (Scrum master)  | See classification response of alternative classifiers section  |


# Classification response of alternative classifiers

See `ml/models/research/A6/model_response_time.ipynb` for experiment results and setup.

The purpose of this task is to test the response time of different classifiers. The included classifiers are presented in the table below: 

| # | Classifier Name             | Notes                                                                                         |
| - | --------------------------- | --------------------------------------------------------------------------------------------- |
| 1 | logistic_regression         | Logistic Regression classifier                                                                |
| 2 | knn5                        | K-Nearest Neighbours classifier using k = 5                                                   |
| 3 | lightgbm                    | Light Gradient Boosting Machine model                                                         |
| 4 | gaussian_nb                 | Gaussian Naive Bayes classifier                                                               |
| 5 | lda                         | Linear Discriminant Analysis classifier                                                       |
| 6 | svm                         | Support Vector Machine classifier                                                             |
| 7 | hist_gradient_boosting_tree | Histogram-based Gradient Boosting Tree classifier |


## Experiment setup

The classifiers in the table are loaded from MLFlow and inserted in a python dictionary along with a prefixed sized array. The purpose of the array is to store time measurments of different predictions made by a particular model. The measurments are recorded by simply capturing timestamps before and after a prediction is made and compute the time difference as shown in CODE 1:

```python
# CODE 1
for model_name, (model, times) in models.items():

    # cache warmup
    warm_row = data_frame.loc[[0]]
    for _ in range(warm_up_rounds):
        model.predict(warm_row)
        
    # Experiment execution
    for i in range(runs): # runs = 2000
        row = data_frame.loc[[i]]
        start = time.perf_counter()
        model.predict(row)
        stop = time.perf_counter()

        diff = (stop - start) * 1000 # ms
        times[i] = diff
```

To allow fair comparisons between models, each model uses the same feature input values to make predictions. These are randomly generated and saved inside a pandas DataFrame. A pre-defined seed value is used to allow reproducibility. For each model, time measurements of model predictions are made and saved for all rows in the DataFrame. However, before any measurments are made, caches are warmed up such that memory call time are approximately constant for each run.

## Results

Performance measurements:
| # | Classifier Name              | mean (ms) | std (ms) |
|---|------------------------------|-----------|----------|
| 1 | logistic_regression          | 0.380     | 0.085    |
| 2 | knn5                         | 1.156     | 0.402    |
| 3 | lightgbm                     | 0.991     | 0.718    |
| 4 | gaussian_nb                  | 0.446     | 0.046    |
| 5 | lda                          | 0.373     | 0.064    |
| 6 | svm                          | 0.542     | 0.195    |
| 7 | hist_gradient_boosting_tree  | 125.755   | 45.529   |

The table above display the means and standard devations derived from the time measurments of model predictions. The image below shows how different means and standard deviations differ between models. The mean and standard deviation for hist_gradient_boosting_tree is not visible as its mean is too large to be visible in the diagram.

![alt text](img/prediction_time_comparison_between_models.png)


### ML SVM Classification

In this section, the work in `svm_classification.ipynb` follows an iterative ML process where the main change from earlier classification experiments is the introduction of Support Vector Machines (SVM) as an alternative model family for the WeakestLink classification endpoint. The first iteration focues on constructing the target lable consistently with earlier notebooks by defining **WeakestLink** as the indicator column with the maximum value per row. During this step it became clear that rows with multiple maxima introduce ambiguity, so those tie cases were removed to keep the lable definition deterministic. A further issue emerged when attempting a split and cross-validation. Some classes can occur too rarely for stratifiction to work reliable. To address this, the pipeline was aligned with the earlier notebooks by dropping classes with too few samples so that the train/test split and k-fold procedure remain valid and stable.

After the label and dataset were stabilized, the preprocessing and feature handling were adapted to SVM requirements and to the endpoint constraints. SVM models are sensitive to feature scale, so the final training pipeline includes median imputation for missing values and standardization before fitting the classifier. To ensure that the evaluation is comparable to the currently deployed approach, the feature set was aligned to the champion´s expected feature ordering using the feature column list stored in MLFlow metadata. This avoids comparing models on mismatched input spaces and ensures that the challenger model remains compatible with the same endpoint schema.

The core modeling iteration implements grid optimization to systematically explore SVM kernels and hyperparameters. Multiple kernels (linear, RBF, polynomial, and sigmoid) were included because each corresponds to different assumtpions about decision boundary complexity, and the optimal choice is data-dependent. Hyperparameters such as `C`, `gamma`, polynomial `degree`, and `coef0`, as well as optional class weighting were searched using GridSearchCV with stratified k-fold cross valdiation. Balanced accuracy was used as the primary optimication metric during cross-validation due to the imbalance in WeakestLink labels, since it reduces the risk of selecting a model that performs well primarily by favoring the largest classes. A practical iteration was required to manage runtime, enabling probabilites inside `SVC` significantly increases training cost because probability estimates required additional internal fitting. To keep grid optimization feasible, cross-validation was run with `probability=False`, and only the final selected SVM, configuration was refit once with `probability=True` to enable `predict_proba()` for log-loss and probabiltiy-based reporting.

Once the best SVM configuration was selected, it was evaluated on a held-out test split using accuracy, balanced accuracy, macro F1, weighted F1, and log-loss where probabilites were available. The final step of the notebook challenges the current champion classification by loading the champion from the MLFlow Model Registry and comparing it to the tuned SVM under the same evaluation setup and aligned feature columns. This comparison is then logged and docuemted so that the decision about whether the SVM should replace the current champion is based on tracked, reproducible evidence rather than ad-hoc reruns. Throughout the notebook, MLFlow is used to record paramters, cross-validation results, test metrics, runtimes, and artifacts (including test inputs and prediction ouitputs) so that the SVM challenger experiments are fully reproducible and auditable, and so that the iteration history over data handling, optimization strategy, and evaluation is explicitly preserve.